**Code-only release candidate.** Market inputs and executed outputs are deliberately absent pending redistribution clearance. See `DATA_SCHEMA.md` before running market-dependent cells. `python src/teaching_interfaces.py` runs deterministic synthetic checks without market data; it does not reproduce the empirical results.

# Cross-SDK, Shot and Noise Validation

## Goal

Companion to Chapter 4. Quantify what changes when a logical financial circuit is represented, compiled, measured or subjected to a specified noise model. Run the portfolio and learning notebooks first.

## Setup

Run this notebook from top to bottom. The experiment uses local simulation only; no cloud account, trading connection or API key is needed. Market observations stop on **31 August 2026**. The supplied data are quoted-price proxies, not verified total-return series.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

# Resolve only the package itself; no author-specific absolute path is needed.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "daily_market.csv").exists(), "Authorised daily inputs are required; see DATA_SCHEMA.md. Synthetic checks run without market data via src/teaching_interfaces.py."
sys.path.insert(0, str(ROOT / "src"))
from common import DATA, RESULTS, market


## 1. Run the controlled implementation studies

The Qiskit topology study compares all-to-all and line connectivity. Kernel-shot experiments hold market inputs fixed. Mixed-state noise experiments use 24 fixed pairs, not a claimed physical device calibration.

In [ ]:
assert (RESULTS / "portfolio_parameters.json").exists()
assert (RESULTS / "qml_kernel_inputs.npz").exists()
from hardware_validation import run
summary = run()
print("Maximum SDK probability difference:", summary["max_sdk_probability_error"])
display(pd.read_csv(RESULTS / "hardware_compilation.csv").drop(columns="probability_error"))

## 2. Assess measurement and spectral effects

Independent shot estimates can make a kernel indefinite. Positive spectral landmarks supply both a repaired training representation and a defined out-of-sample map. Seed standard deviations here are not uncertainty about the stock-market population.

In [ ]:
display(pd.DataFrame(summary["shot_summary"]).round(5))
display(Image(filename=str(RESULTS / "hardware_sensitivity.png")))

## 3. Separate circuit simplification from sampling

The reduced overlap circuit removes terminal cancelling entanglers before inserting noise. Unit angular scale is used for these 24-pair diagnostics and matches the selected classifier's angular scale, but depths and noise conditions are controlled separately. This preserves ideal overlaps but lowers channel exposure. More shots would not remove this channel bias.

In [ ]:
noise = pd.read_csv(RESULTS / "hardware_noise_kernel.csv")
display(noise[noise.eta == .01][["depth", "cancelled", "rmse", "max_error"]].round(5))

## 4. Calculate post-quantum object sizes

PQC runs on classical computers and protects a different layer. This cell adds object lengths from FIPS 203--205. It does not implement cryptography, measure latency or assert that a protocol is secure. Distinguish KEM ciphertexts, signatures and public keys; certificates and framing are not included.

In [ ]:
from teaching_cases import pqc_object_sizes
sizes = pqc_object_sizes()
display(pd.DataFrame(sizes["objects"]))
print("One KEM key/ciphertext plus one signature key/signature:", sizes["kem_key_and_ciphertext"] + sizes["dsa_key_and_signature"], "bytes")
from teaching_interfaces import smoke_test
print(smoke_test())

## Interpretation and Next Steps

Cross-SDK probabilities agree numerically after bit-order correction. Compare the eight-qubit row under both topologies to see the routing overhead. All circuit results remain local simulations. Physical-device validation needs target-specific compilation, calibration metadata, actual counts and timing. PQC object sizes are derived quantities, not a deployment test. The chapter supplies a proposed authenticated-result teaching exercise, clearly distinguished from what this notebook executes.